In [7]:
%pip install statsmodels


import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

ROOT = Path.cwd().resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

def fm_regression(df, ycol, xcols, datecol="date"):
    out = []
    for d, g in df.dropna(subset=[ycol] + xcols).groupby(datecol):
        if len(g) >= len(xcols) + 5:
            X = sm.add_constant(g[xcols])
            y = g[ycol]
            res = sm.OLS(y, X).fit()
            row = {"date": d}
            for k, v in res.params.items():
                row[k] = v
            out.append(row)
    return pd.DataFrame(out)

rets = pd.read_csv(DATA_PROCESSED / "return_panel_500.csv", parse_dates=["date"])
rets["ticker"] = rets["ticker"].astype(str)
rets = rets.sort_values(["ticker", "date"]).reset_index(drop=True)
rets["ret_fwd"] = rets.groupby("ticker")["return"].shift(-1)

a = pd.read_csv(OUT / "factor_A_value_358.csv")
aq = pd.read_csv(OUT / "factor_A_quality_358.csv")
asue = pd.read_csv(OUT / "factor_A_sue_358.csv")
am = pd.read_csv(OUT / "factor_A_momentum_358.csv", parse_dates=["date"])

a = a.merge(aq[["ticker", "report_date", "quality_score"]], on=["ticker", "report_date"], how="inner")
a = a.merge(asue[["ticker", "report_date", "sue_z"]], on=["ticker", "report_date"], how="inner")
am = am.rename(columns={"date": "report_date"})
a["report_date"] = pd.to_datetime(a["report_date"])
am["report_date"] = pd.to_datetime(am["report_date"])
a = a.merge(am[["ticker", "report_date", "momentum_12m_z"]], on=["ticker", "report_date"], how="inner")
a = a.rename(columns={"report_date": "date"})
a["date"] = pd.to_datetime(a["date"])

a = a.merge(rets[["date", "ticker", "ret_fwd"]], on=["date", "ticker"], how="inner")
fm_a = fm_regression(a, "ret_fwd", ["value_score", "quality_score", "sue_z", "momentum_12m_z"], "date")

b = pd.read_csv(OUT / "factor_B_price_only_500.csv", parse_dates=["date"])
b = b.merge(rets[["date", "ticker", "ret_fwd"]], on=["date", "ticker"], how="inner")
fm_b = fm_regression(b, "ret_fwd", ["momentum_12m_z"], "date")

cv = pd.read_csv(OUT / "factor_C_value_overlap.csv")
cq = pd.read_csv(OUT / "factor_C_quality_overlap.csv")
cs = pd.read_csv(OUT / "factor_C_sue_overlap.csv")
cm = pd.read_csv(OUT / "factor_C_momentum_overlap.csv", parse_dates=["date"])

c = cv.merge(cq[["ticker", "report_date", "quality_score"]], on=["ticker", "report_date"], how="inner")
c = c.merge(cs[["ticker", "report_date", "sue_z"]], on=["ticker", "report_date"], how="inner")
cm = cm.rename(columns={"date": "report_date"})
c["report_date"] = pd.to_datetime(c["report_date"])
cm["report_date"] = pd.to_datetime(cm["report_date"])
c = c.merge(cm[["ticker", "report_date", "momentum_12m_z"]], on=["ticker", "report_date"], how="inner")
c = c.rename(columns={"report_date": "date"})
c["date"] = pd.to_datetime(c["date"])
c = c.merge(rets[["date", "ticker", "ret_fwd"]], on=["date", "ticker"], how="inner")
fm_c = fm_regression(c, "ret_fwd", ["value_score", "quality_score", "sue_z", "momentum_12m_z"], "date")

def summarize(fm):
    cols = [c for c in fm.columns if c != "date"]
    s = []
    for col in cols:
        mean = fm[col].mean()
        se = fm[col].std(ddof=1) / np.sqrt(len(fm))
        t = mean / se if se != 0 else np.nan
        s.append({"factor": col, "mean_coef": mean, "t_stat": t, "n_months": len(fm)})
    return pd.DataFrame(s)

sum_a = summarize(fm_a)
sum_b = summarize(fm_b)
sum_c = summarize(fm_c)

fm_a.to_csv(OUT / "fama_macbeth_A.csv", index=False)
fm_b.to_csv(OUT / "fama_macbeth_B.csv", index=False)
fm_c.to_csv(OUT / "fama_macbeth_C.csv", index=False)

sum_a.to_csv(OUT / "fama_macbeth_A_summary.csv", index=False)
sum_b.to_csv(OUT / "fama_macbeth_B_summary.csv", index=False)
sum_c.to_csv(OUT / "fama_macbeth_C_summary.csv", index=False)

print(sum_a)
print(sum_b)
print(sum_c)


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
           factor  mean_coef    t_stat  n_months
0           const   0.011245  1.337572        47
1     value_score  -0.001883 -0.583992        47
2   quality_score   0.006537  0.770644        47
3           sue_z  -0.001151 -0.264271        47
4  momentum_12m_z   0.008022  1.155534        47
           factor  mean_coef    t_stat  n_months
0           const   0.013387  3.346759       135
1  momentum_12m_z   0.002758  2.015617       135
           factor  mean_coef    t_stat  n_months
0           const   0.011246  1.349799        47
1     value_score  -0.001906 -0.593680        47
2   quality_score   0.006434  0.760430        47
3           sue_z  -0.001127 -0.258642        47
4  momentum_12m_z   0.007973  1.149650        47
